# 05 - Data Preprocessing

Prepare the cleaned match data for machine learning: handle remaining missing values, encode categorical variables, build the target variable, split into X/y, and scale numerical features.

Target: `team1_won` (1 if `team1` won, 0 if `team2` won). Matches with no decisive result (tie / no result) are dropped since there's no winner to predict.

In [1]:
import os
import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

matches = pd.read_csv('../data/processed/matches_cleaned.csv', parse_dates=['date'])
matches.shape

(1212, 11)

## 1. Handle remaining missing values

Drop matches with no decisive winner (tie / no result / abandoned) since the target can't be defined for them. Fill any remaining gaps in feature columns.

In [2]:
print('rows before dropping non-decisive matches:', len(matches))
matches = matches[
    matches['winner'].notna()
    & ((matches['winner'] == matches['team1']) | (matches['winner'] == matches['team2']))
].copy()
print('rows after dropping non-decisive matches:', len(matches))

for col in ['city', 'venue', 'toss_winner', 'toss_decision']:
    matches[col] = matches[col].fillna('Unknown')

rows before dropping non-decisive matches: 1212
rows after dropping non-decisive matches: 1187


## 2. Create the target variable

In [3]:
matches['team1_won'] = (matches['winner'] == matches['team1']).astype(int)
matches['team1_won'].value_counts(normalize=True)

team1_won
1    0.502106
0    0.497894
Name: proportion, dtype: float64

## 3. Encode categorical variables

`team1`, `team2` and `toss_winner` share the same set of team names, so they're fit on the combined vocabulary with one shared encoder. Each encoder is saved for reuse at inference time.

In [4]:
team_encoder = LabelEncoder()
all_teams = pd.concat([matches['team1'], matches['team2'], matches['toss_winner']]).unique()
team_encoder.fit(all_teams)

matches['team1_enc'] = team_encoder.transform(matches['team1'])
matches['team2_enc'] = team_encoder.transform(matches['team2'])
matches['toss_winner_enc'] = team_encoder.transform(matches['toss_winner'])

venue_encoder = LabelEncoder()
matches['venue_enc'] = venue_encoder.fit_transform(matches['venue'])

city_encoder = LabelEncoder()
matches['city_enc'] = city_encoder.fit_transform(matches['city'])

toss_decision_encoder = LabelEncoder()
matches['toss_decision_enc'] = toss_decision_encoder.fit_transform(matches['toss_decision'])

joblib.dump(team_encoder, f'{MODELS_DIR}/team_encoder.joblib')
joblib.dump(venue_encoder, f'{MODELS_DIR}/venue_encoder.joblib')
joblib.dump(city_encoder, f'{MODELS_DIR}/city_encoder.joblib')
joblib.dump(toss_decision_encoder, f'{MODELS_DIR}/toss_decision_encoder.joblib')

['../models/toss_decision_encoder.joblib']

## 4. Split features (X) and target (y)

In [5]:
feature_cols = [
    'team1_enc', 'team2_enc', 'venue_enc', 'city_enc',
    'toss_winner_enc', 'toss_decision_enc', 'season',
]

X = matches[feature_cols].copy()
y = matches['team1_won'].copy()

X.shape, y.shape

((1187, 7), (1187,))

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((949, 7), (238, 7))

## 5. Scale numerical features

`season` is the only feature on a different numeric scale from the label-encoded IDs, so it's standardized. The scaler is fit on the training split only, to avoid leaking test-set statistics.

In [7]:
scaler = StandardScaler()
X_train['season_scaled'] = scaler.fit_transform(X_train[['season']])
X_test['season_scaled'] = scaler.transform(X_test[['season']])

joblib.dump(scaler, f'{MODELS_DIR}/season_scaler.joblib')

['../models/season_scaler.joblib']

## Save preprocessed dataset

In [8]:
X_train.assign(team1_won=y_train.values).to_csv('../data/processed/train.csv', index=False)
X_test.assign(team1_won=y_test.values).to_csv('../data/processed/test.csv', index=False)

print('train:', X_train.shape, 'test:', X_test.shape)

train: (949, 8) test: (238, 8)
